# AIMO 3 Notebook 1

## Workspace Configuration 

In [1]:
# Installs Packages where not already installed 

import sys
import subprocess
import importlib.util

required_packages = ["transformers", "accelerate", "torch"]
missing_packages = [p for p in required_packages if importlib.util.find_spec(p) is None]

if missing_packages:
    print(f"Installing missing packages: {missing_packages}")
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing_packages])
else:
    print("All required packages are already installed.")

All required packages are already installed.


In [2]:
import torch
import numpy as np
#assert torch.__version__ == "2.8.0+cu128", (f"Torch version is {torch.__version__} instead of 2.8.0+cu128")

print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("CUDA version:", torch.version.cuda)
print("GPU count:", torch.cuda.device_count())
print("NUMPY version", np.__version__)

if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))

#  assert np.__version__ == "2.2.0", f"Numpy version is {np.__version__} instead of 2.2.0"
assert torch.cuda.is_available and torch.cuda.device_count() >= 1, "GPU not enabled"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

Torch version: 2.6.0+cu124
CUDA available: True
CUDA version: 12.4
GPU count: 2
NUMPY version 1.26.4
GPU name: Tesla T4


## Data Loaders

In [25]:
# Data Loader Configurations 
# Competition Context
_RULES_FORMAT_NOTATION = r"""## Mathematical Olympiad Formatting & Notation Rules
### LaTeX Environment
- Use LaTeX for all mathematical notation. Permitted packages: `amssymb`, `amsmath`, `enumitem` only.
- Equation environments: `$...$`, `\(...\)`, `\[...\]`, `\begin{equation}...\end{equation}`, `\begin{equation*}...\end{equation*}`
- Lists: `\itemize` and `\enumerate` (use `enumitem` conventions)
- No other LaTeX packages or environments are permitted.

### Variables & Symbols
- Variables: lowercase latin letters e.g. $a, x, y$; calligraphic letters via `\mathcal` e.g. $\mathcal{R}, \mathcal{S}$
- Small integers (1, 2, 3) may appear outside math mode; large integers like $10^{50}$ must be in math mode.
- Sequences may be continued with `\ldots`, `\cdots`, or `...`

### Arithmetic & Operations
- Multiplication: simple adjacency $2x$, or `\cdot` for $x \cdot y$, or `\times` for $x \times y$
- Factorial: $n! = n \cdot (n-1) \cdot (n-2) \cdots 2 \cdot 1$, with convention $0! = 1$
- Fractions: display form $\frac{a}{b}$ or inline $a/b$
- Absolute value: $|x|$, e.g. $|-2| = 2$
- Percentages: $x\%$ represents the value $\frac{x}{100}$

### Functions & Logarithms
- Standard functions $\log, \sin, \cos$ are in math mode by default.
- Unless a base is stated explicitly (e.g. $\log_2 x$), $\log$ means the natural logarithm.

### Floor, Ceiling & Fractional Parts
- $\lfloor x \rfloor$ = greatest integer ≤ $x$ (floor)
- $\lceil x \rceil$ = smallest integer ≥ $x$ (ceiling)
- $\{x\}$ = fractional part of $x$, i.e. $x - \lfloor x \rfloor$
- Use `\left` and `\right` for auto-sizing, e.g. `\left\lceil \frac{x}{2} \right\rceil`

### Overline Notation
- $\overline{abcd}$ denotes the integer formed by digits $a, b, c, d \in \{0,1,\ldots,9\}$ with $a \neq 0$
- Example: $a=1, b=2$ gives $\overline{ab} = 12$; $\overline{bab} = 212$

### Combinatorics
- $\binom{a}{b}$ = binomial coefficient = number of ways to choose $b \geq 0$ elements from a set of size $a \geq 0$
- Convention: $\binom{a}{0} = 1$ and $\binom{a}{b} = 0$ if $b > a$

### Number Sets
- $\mathbb{N}$ = positive integers (i.e. $> 0$)
- $\mathbb{Z}$ = integers
- $\mathbb{Q}$ = rational numbers
- $\mathbb{R}$ = real numbers
- $\mathbb{Z}_{\geq n}$ = integers $\geq n$

### Functions & Sets
- Function notation: $f: S \to T$ where $S$ and $T$ are sets
- Sets use standard curly-bracket notation e.g. $\{1, 2, 3\}$
- Sum over empty set = $0$; product over empty set = $1$
- A point "chosen uniformly at random" from interval $[0,1]$ is standard probability language.

### Geometry
- Triangle notation: $\triangle ABC$ with sides $AB$, $BC$, $AC$
- Angle notation: $\angle ABC$; degrees written as $\angle ABC = 60°$
- Measures with no units are assumed to be in radians.
- Triangle altitudes: the altitude from vertex $A$ may be called "the $A$-altitude" or "$h_a$"; similarly for median lines.
- Taxonomy: equilateral triangles are isosceles; squares are rectangles; rectangles are parallelograms.
- "Trapezium" = quadrilateral with at least one pair of parallel opposite sides (British/Australian definition).

### Indexing & Sequences
- For integers $l$, $m$, $n$: $l^{m^n}$ denotes $l^{(m^n)}$ (right-associative tower)

### Definitions
- `:=` means "is defined to be equal to"

### Language
- British or American English are both acceptable. "Highest common factor" = "greatest common divisor".
"""

_RULE_RESPONSE_FORMAT = r"""
## Answer Format Rules (AIMO2 Style)
- All answers must be integers in the range $[0, 99999]$ inclusive.
- All questions are worded so the final answer lies in this range; any required modulus is stated explicitly in the problem.
- For "remainder when $N$ is divided by $q$" questions: return the unique integer $r$ such that $0 \leq r < q$ and $q \mid N - r$.
- Answers may require basic computation including modular arithmetic.
  - Example: $\lfloor 10^4 \sqrt{2} \rfloor = 14142$
  - Example: $3^{2025} \equiv 29443 \pmod{10^5}$
- Report only the final integer answer; do not include units, variables, or expressions."""

# Example Problem Data Samples
EXAMPLE_PROBLEM = r"Define a function $f \\colon \\mathbb{Z}_{\\geq 1} \\to \\mathbb{Z}_{\\geq 1}$ by\n\\begin{equation*}\n    f(n) = \\sum_{i = 1}^n \\sum_{j = 1}^n j^{1024} \\left\\lfloor\\frac1j + \\frac{n-i}{n}\\right\\rfloor.\n\\end{equation*}\nLet $M=2 \\cdot 3 \\cdot 5 \\cdot 7 \\cdot 11 \\cdot 13$ and let $N = f{\\left(M^{15}\\right)} - f{\\left(M^{15}-1\\right)}$. Let $k$ be the largest non-negative integer such that $2^k$ divides $N$. What is the remainder when $2^k$ is divided by $5^7$?"
EXAMPLE_ANSWER = "32951"
EXAMPLE_PROBLEM_RESPONSE = r"Let $ABC$ be an acute-angled triangle with integer side lengths and $AB<AC$. Points $D$ and $E$ lie on segments $BC$ and $AC$, respectively, such that $AD=AE=AB$. Line $DE$ intersects $AB$ at $X$. Circles $BXD$ and $CED$ intersect for the second time at $Y \\neq D$. Suppose that $Y$ lies on line $AD$. There is a unique such triangle with minimal perimeter. This triangle has side lengths $a=BC$, $b=CA$, and $c=AB$. Find the remainder when $abc$ is divided by $10^{5}$."
EXAMPLE_ANSWER_RESPONSE = "336"
SYS_PROMPT = """You generate AIMO-style olympiad math problems.

Rules:
- The problem must be original and similar in style to AIME/AIMO.
- Use concise olympiad wording and standard mathematical notation.
- The problem must have exactly one final integer answer.
- The answer must be a non-negative integer.
- Do not add explanations, steps, or commentary.
- Do not make the problem trivial or too long.
- Keep the topic and difficulty close to the original problem.
- Return valid JSON only.
"""


USER_INPUT_TEMPLATE = """**Problem**
{problem}
**Answer**
{answer}

Create one new AIMO-style variant with the same topic but of difficulty level: {challenge_level}.

Return JSON exactly in this format:
{{
  "variant_problem": "...",
  "variant_answer": "12345"
}}
"""

In [32]:
# Data Loader Excecute
import pandas as pd 


CHALLENGE_LEVELS = ["easy", "moderate", "difficult"]

class AIMOTrainDataset:
    def __init__(self, load_path: str = "/kaggle/input/ai-mathematical-olympiad-progress-prize-3/reference.csv"):
        self.df = pd.read_csv(load_path) 
        self.preprocess()
    
    def preprocess(self):
        df = self.df.copy()

        # Create Cartesian product (expand rows)
        df = df.loc[df.index.repeat(len(CHALLENGE_LEVELS))].reset_index(drop=True)

        # Assign variant_mode cyclically
        df["variant_mode"] = CHALLENGE_LEVELS * (len(df) // len(CHALLENGE_LEVELS))

        # Construct inputs with challenge level
        df["inputs"] = df.apply(
            lambda x: USER_INPUT_TEMPLATE.format(
                problem=x["problem"],
                answer=x["answer"],
                challenge_level=x["variant_mode"]
            ),
            axis=1
        )

        # Assign back
        self.df = df

        assert {'id', 'problem', 'answer', 'inputs', 'variant_mode'}.issubset(set(self.df.columns))
        
class AIMODataset:
    def __init__(self):
        self.train = AIMOTrainDataset()

    def __repr__(self):
        return (
            f"AIMODataset Summary\n"
            f"{'-'*40}\n"
            f"Train shape : {self.train.df.shape}\n"
            f"Train columns:\n{self.train.df.dtypes.to_string()}\n\n"
            f"{self.train.df.head(5)}"
            f"\n\nExample `inputs` field:\n\n {self.train.df.inputs.iloc[0].strip()}"
        )
        
data = AIMODataset()
print(data)

AIMODataset Summary
----------------------------------------
Train shape : (30, 5)
Train columns:
id              object
problem         object
answer           int64
variant_mode    object
inputs          object

       id                                            problem  answer  \
0  0e644e  Let $ABC$ be an acute-angled triangle with int...     336   
1  0e644e  Let $ABC$ be an acute-angled triangle with int...     336   
2  0e644e  Let $ABC$ be an acute-angled triangle with int...     336   
3  26de63  Define a function $f \colon \mathbb{Z}_{\geq 1...   32951   
4  26de63  Define a function $f \colon \mathbb{Z}_{\geq 1...   32951   

  variant_mode                                             inputs  
0         easy  **Problem**\nLet $ABC$ be an acute-angled tria...  
1     moderate  **Problem**\nLet $ABC$ be an acute-angled tria...  
2    difficult  **Problem**\nLet $ABC$ be an acute-angled tria...  
3         easy  **Problem**\nDefine a function $f \colon \math...  
4     moderat

## Pretrained Models

In [ ]:
# Load the Model with Hugging Face
import os

os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
os.environ["TRANSFORMERS_VERBOSITY"] = "debug"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

from transformers import AutoModelForCausalLM, AutoTokenizer

# parent model name = "Qwen/Qwen3-30B-A3B"
MODEL_PATH = "/kaggle/input/models/qwen-lm/qwen-3/transformers/4b-base/1"

# load the tokenizer and the model
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, local_files_only=True)
model = AutoModelForCausalLM.from_pretrained(
            MODEL_PATH,
            torch_dtype="auto",   
            device_map='auto',
            local_files_only=True  
    )

print(f"Completed loading model from path {MODEL_PATH}")

JAX version 0.5.2, Flax version 0.10.6 available.
loading file vocab.json
loading file merges.txt
loading file tokenizer.json
loading file added_tokens.json
loading file special_tokens_map.json
loading file tokenizer_config.json
loading file chat_template.jinja
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.
loading configuration file /kaggle/input/models/qwen-lm/qwen-3/transformers/4b-base/1/config.json
Model config Qwen3Config {
  "architectures": [
    "Qwen3ForCausalLM"
  ],
  "attention_bias": false,
  "attention_dropout": 0.0,
  "bos_token_id": 151643,
  "eos_token_id": 151643,
  "head_dim": 128,
  "hidden_act": "silu",
  "hidden_size": 2560,
  "initializer_range": 0.02,
  "intermediate_size": 9728,
  "layer_types": [
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention"

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

loading weights file /kaggle/input/models/qwen-lm/qwen-3/transformers/4b-base/1/model.safetensors.index.json
Instantiating Qwen3ForCausalLM model under default dtype torch.bfloat16.
Generate config GenerationConfig {
  "bos_token_id": 151643,
  "eos_token_id": 151643
}



Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

All model checkpoint weights were used when initializing Qwen3ForCausalLM.

All the weights of Qwen3ForCausalLM were initialized from the model checkpoint at /kaggle/input/models/qwen-lm/qwen-3/transformers/4b-base/1.
If your task is similar to the task the model of the checkpoint was trained on, you can already use Qwen3ForCausalLM for predictions without further training.
loading configuration file /kaggle/input/models/qwen-lm/qwen-3/transformers/4b-base/1/generation_config.json
Generate config GenerationConfig {
  "bos_token_id": 151643,
  "eos_token_id": 151643,
  "max_new_tokens": 2048
}



Completed loading model from path /kaggle/input/models/qwen-lm/qwen-3/transformers/4b-base/1


## Model Main Entry Point 

In [37]:
# Complete Script to load the main model with Hugging Face Transformers

from typing import Literal
from datetime import datetime 

DEFAULT_DECODE_KWARGS = {
    "max_new_tokens": 512,
    "do_sample": False,
    #"num_beams": 0,
    #"top_k": 90,
    #"top_p": 0.95, 
    "use_cache": False,
    "pad_token_id": tokenizer.eos_token_id,
    
}

class QwenDataAgent:
    # parent model name = "Qwen/Qwen3-30B-A3B"
    """ Data Agent to prepare data batches to stream to base model. 
    
    NOTES:
        To empty cache: 
            torch.cuda.empty_cache()
        To view cuda memory summary:
            torch.cuda.memory_summary()
    """
    def __init__(self, tokenizer, model):
        self.tokenizer = tokenizer
        self.model = model
        self.decode_kwargs = DEFAULT_DECODE_KWARGS.copy()
        self.system_prompt = SYS_PROMPT
        self.responses = []
        
        self.model.config.use_cache = False
        self.model.eval()
        
    def generate(self, row):

        try:
            inputs = [
                {"role": "system", "content": self.system_prompt },
                {"role": "user", "content": row['inputs'] }
            ]

            text = self.tokenizer.apply_chat_template(
                inputs,
                tokenize=False,
                add_generation_prompt=True,
                enable_thinking=False
            )

            model_inputs = self.tokenizer([text], return_tensors="pt")
            model_inputs = {k: v.to("cuda") for k, v in model_inputs.items()}
            print(f"Input tensor allocated device: {model_inputs['input_ids'].device}")

            # conduct text completion
            generated_ids = self.model.generate(
                **model_inputs,
                **self.decode_kwargs
            )
            output_ids = generated_ids[0][len(model_inputs['input_ids'][0]):].tolist() 
            print(f"Finished generating response IDs allocated device: {generated_ids.device}")
            
            # rindex finding 151668 (</think>)
            try:
                index = len(output_ids) - output_ids[::-1].index(151668)
            except ValueError:
                index = 0
                
            #thinking_content = self.tokenizer.decode(output_ids[:index], skip_special_tokens=True).strip("\n")
            content = self.tokenizer.decode(output_ids[index], skip_special_tokens=True).strip("\n")
            
            #print("thinking content:", thinking_content)
            print("content:", content)
            
            #match = _USER_INPUT_PATTERN.search(content)
            #if match:
            #    problem = match.group("problem").strip()
            #    answer = match.group("answer").strip()
            #else:
            #    problem, answer = None, None  # or raise if strict
                
            response = {
                "timedate": datetime.now(),
                "inputs_id": row['id'],
                "inputs": row['inputs'],
                "challenge_level": row['variant_mode'],
                #"thinking": thinking_content,
                #"problem": problem,
                #"answer": answer,
                "_raw_content": content,
            }
            
            self.responses.append(response)
            return response 
            
        except Exception as e:
            raise e

qwe = QwenDataAgent(tokenizer=tokenizer, model=model)

print("Finished configuring models")
print("Trialing workflow with sample row")
qwe.generate(row=data.train.df.iloc[1])

history = pd.DataFrame.from_records(qwe.responses)
history

Finished configuring models
Trialing workflow with sample row
Input tensor allocated device: cuda:0
Finished generating response IDs allocated device: cuda:0
content: {


,timedate,inputs_id,inputs,challenge_level,_raw_content
0,2026-04-02 19:08:31.316923,0e644e,**Problem**\nLet $ABC$ be an acute-angled tria...,moderate,{


## Comparison & Analytics

In [ ]:
print("Comparing >> Raw Content: \n")
print(history['_raw_content'].iloc[0])

Comparing >> Raw Content: 

**


In [ ]:
print("Comparing >> Inputs Content: \n")
print(history['inputs'].iloc[0])

Comparing >> Inputs Content: 

**Problem**
Define a function $f \colon \mathbb{Z}_{\geq 1} \to \mathbb{Z}_{\geq 1}$ by
\begin{equation*}
    f(n) = \sum_{i = 1}^n \sum_{j = 1}^n j^{1024} \left\lfloor\frac1j + \frac{n-i}{n}\right\rfloor.
\end{equation*}
Let $M=2 \cdot 3 \cdot 5 \cdot 7 \cdot 11 \cdot 13$ and let $N = f{\left(M^{15}\right)} - f{\left(M^{15}-1\right)}$. Let $k$ be the largest non-negative integer such that $2^k$ divides $N$. What is the remainder when $2^k$ is divided by $5^7$?
**Answer**
32951
